In [41]:
import numpy as np
import pandas as pd
from scipy.stats import norm

predicted_rpace = pd.read_csv("predicted_rpace.csv", index_col=0).squeeze('columns')
current_qpace = pd.read_csv("current_qpace.csv", index_col=0).squeeze('columns')
results_archive = pd.read_csv("data/races/results_archive.csv")

current_year = 2026

drivers = sorted(predicted_rpace.index) # canonical order, used everywhere below
D = len(drivers)
N = 10000

mu_r = predicted_rpace['mu'].reindex(drivers).values
sigma_r = predicted_rpace['sigma'].reindex(drivers).values
mu_q = current_qpace['mu'].reindex(drivers).values
sigma_q = current_qpace['sigma'].reindex(drivers).values

teams = [
    results_archive[(results_archive['Abbreviation'] == d) & (results_archive['Year'] == current_year)].iloc[0]['TeamName']
    for d in drivers
]
team_list = sorted(set(teams))
team_index = np.array([team_list.index(t) for t in teams])
n_teams = len(team_list)

rng = np.random.default_rng()  
predicted_rpace

,mu,sigma
RUS,0.002427,0.002187
VER,0.003051,0.002181
LEC,0.004722,0.002155
PIA,0.005139,0.002187
NOR,0.005557,0.002178
HAM,0.007318,0.002185
ANT,0.007395,0.002169
HAD,0.010057,0.002170
GAS,0.014218,0.002191
HUL,0.019216,0.002179


In [2]:
# BUILD P_DNF
def rolling_dnf_counts(results_archive, current_round, current_year, window=20):
    # Window: most number of recent race entries to look back over per driver
    hist = results_archive[
        (results_archive['Year'] < current_year) | ((results_archive['Year'] == current_year) & (results_archive['Round'] < current_round))
    ].sort_values(['Year', 'Round'])

    counts = (
        hist.groupby('Abbreviation').tail(window)
            .groupby('Abbreviation')['dnf'].agg(dnfs='sum', starts='count')
    )
    return counts

def rolling_team_dnf_counts(results_archive, current_round, current_year, window=40):
    hist = results_archive[
        (results_archive['Year'] < current_year) | ((results_archive['Year'] == current_year) & (results_archive['Round'] < current_round))
    ].sort_values(['Year', 'Round'])

    counts = (
        hist.groupby('TeamId').tail(window)
            .groupby('TeamId')['dnf'].agg(dnfs='sum', starts='count')
    )
    return counts

# blend driver and team beta estimates in the same inverse-variance way as everything else to keep consistent
def beta_dnf_prob(dnfs, starts, alpha, beta):
    a_post = dnfs + alpha
    b_post = (starts - dnfs) + beta
    mean = a_post / (a_post + b_post)
    var = (a_post * b_post) / ((a_post + b_post)**2 * (a_post + b_post + 1))
    return mean, var

# alpha=2, beta=25 (driver) mean prior mean DNF rate of 2/27 or 7.4%
def blended_dnf_prob(driver, team, driver_counts, team_counts, alpha_d=2, beta_d=25, alpha_t=3, beta_t=35):
    if driver in driver_counts.index:
        d_dnfs, d_starts = driver_counts.loc[driver, ['dnfs', 'starts']]
    else:
        d_dnfs, d_starts = 0, 0
    d_mean, d_var = beta_dnf_prob(d_dnfs, d_starts, alpha_d, beta_d)

    if team in team_counts.index:
        t_dnfs, t_starts = team_counts.loc[team, ['dnfs', 'starts']]
    else:
        t_dnfs, t_starts = 0, 0
    t_mean, t_var = beta_dnf_prob(t_dnfs, t_starts, alpha_t, beta_t)

    w_d, w_t = 1.0 / d_var, 1.0 / t_var
    blended = (w_d * d_mean + w_t * t_mean) / (w_d + w_t)
    return float(np.clip(blended, 0.005, 0.5))

current_round = 13
current_year = 2026

driver_counts = rolling_dnf_counts(results_archive, current_round, current_year)
team_counts = rolling_team_dnf_counts(results_archive, current_round, current_year)

p_dnf = {}
for d in drivers:
    team = results_archive[(results_archive['Abbreviation'] == d) & (results_archive['Year'] == current_year)].iloc[0]['TeamName']
    p_dnf[d] = blended_dnf_prob(d, team, driver_counts, team_counts)

p_dnf_series = pd.Series(p_dnf).reindex(drivers)   # same order as mu_r, mu_q, etc.
p_dnf_array = p_dnf_series.values

In [3]:
def blend_sources(sources, tau):
    # sources: dict name -> (mu, sigma)   sigma = sampling SE, fractional units
    # tau: drift sd added
    names, mu, var = [], [], []
    for name, (m, s) in sources.items():
        if not np.isfinite(m) and np.isfinite(s):
            continue
        names.append(name)
        mu.append(m)
        var.append(s**2 + tau.get(name, 0.0)**2)

    if not names:
        return np.nan, np.inf, {}

    mu, var = np.array(mu), np.array(var)
    w = 1.0 / var
    blended_mu = float(np.sum(w * mu) / w.sum())
    blended_sigma = float(np.sqrt(1.0 / w.sum()))
    return blended_mu, blended_sigma, dict(zip(names, w / w.sum()))

In [37]:
def simulate_quali(mu_q, sigma_q, N, rng, grid_penalties=None):
    q_draw = mu_q[None, :] + sigma_q[None, :] * rng.standard_normal((N, len(mu_q)))
    quali_result = q_draw.argsort(axis=1).argsort(axis=1) + 1

    if grid_penalties:
        grid = apply_grid_penalties(quali_result, grid_penalties, drivers)
    else:
        grid = quali_result.copy()
    return grid, quali_result, q_draw

def apply_grid_penalties(grid, grid_penalties, drivers):
    # grid_penalties is a dict that looks like: {'VER': 10} meaning 10 place grid penalty for Verstappen
    grid = grid.copy()
    for i, d in enumerate(drivers):
        places = grid_penalties.get(d, 0)

        if places == 0:
            continue

        for row in range(grid.shape[0]):
            old_pos = grid[row, i]
            new_pos = min(old_pos + places, grid.shape[1])
            shift_mask = (grid[row] > old_pos) & (grid[row] <= new_pos)
            grid[row, shift_mask] -= 1
            grid[row, i] = new_pos

    return grid

In [5]:
def calibrate_overtake_difficulty(results_archive, track):
    # Regress |finish-grid| or grid-finish against track, returns 0-1 difficulty score
    rows = results_archive[results_archive['Track'] == track]
    avg_position_change = (rows['GridPosition'] - rows['Position']).abs().mean()
    # normalize against the full-calendar range
    all_tracks_avg = results_archive.groupby('Track').apply(
        lambda g: (g['GridPosition'] - g['Position']).abs().mean()
    )
    difficulty = 1.0 - (avg_position_change - all_tracks_avg.min()) / (all_tracks_avg.max() - all_tracks_avg.min())
    return float(np.clip(difficulty, 0.05, 0.95))

track_position_weight = calibrate_overtake_difficulty(results_archive, 'Monza')
track_position_weight

C:\Users\nanub\AppData\Local\Temp\ipykernel_29256\2729805903.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  all_tracks_avg = results_archive.groupby('Track').apply(


0.4482581967213114

In [6]:
def simulate_race(mu_r, sigma_r, grid, team_index, n_teams, p_dnf, track_position_weight, team_dnf_corr, N, rng, n_drivers):
    r_draw = mu_r[None, :] + sigma_r[None, :] * rng.standard_normal((N, n_drivers))
    pace_spread = mu_r.max() - mu_r.min()
    
    r_draw = r_draw + track_position_weight * pace_spread * (grid / n_drivers) # harder to overtake means grid matters more

    # correlated dnfs: shared team-level shock and individual noise
    team_shock = rng.standard_normal((N, n_teams))[:, team_index]
    z = team_dnf_corr * team_shock + np.sqrt(1 - team_dnf_corr**2) * rng.standard_normal((N, n_drivers))
    dnf_threshold = norm.ppf(np.clip(p_dnf, 1e-4, 0.999))[None, :]
    dnf = z < dnf_threshold

    r_draw_scored = np.where(dnf, np.inf, r_draw)
    finish = r_draw_scored.argsort(axis=1).argsort(axis=1) + 1
    finish = np.where(dnf, np.nan, finish) # drivers who dnfed have no finishing position
    return finish, dnf, r_draw

In [7]:
def simulate_fastest_lap(finish, r_draw, dnf, N, n_drivers, top_n=8):
    fl = np.zeros((N, n_drivers), dtype=bool)
    for row in range(N):
        finishers = np.where(~dnf[row])[0]
        if len(finishers) == 0:
            continue

        # fastest lap can be gotten by anyone in top_n finishers by position or anyone with fresher pace
        candidates = finishers[np.argsort(finish[row, finishers])][:top_n]
        pace = -r_draw[row, candidates]
        probs = np.exp(pace * 5) / np.sum(np.exp(pace * 5))
        winner = rng.choice(candidates, p=probs)
        fl[row, winner] = True
    return fl

In [8]:
def simulate_driver_of_the_day(finish, grid, dnf, N, n_drivers):
    positions_gained = np.where(dnf, -999, grid - np.nan_to_num(finish, nan=999))
    dotd = np.zeros((N, n_drivers), dtype=bool)
    logits = positions_gained * 0.3
    for row in range(N):
        valid = positions_gained[row] > -999
        if not valid.any():
            continue
        p = np.exp(logits[row, valid] - logits[row, valid].max())
        p = p / p.sum()
        index = np.where(valid)[0]
        winner = rng.choice(index, p=p)
        dotd[row, winner] = True

    return dotd

In [9]:
def simulate_pit_stops(team_index, n_teams, N, team_pit_mu, team_pit_sigma, rng):
    pit_times = team_pit_mu[None, :] + team_pit_sigma[None, :] * rng.standard_normal((N, n_teams))
    fastest_pit_team = pit_times.argmin(axis=1)
    return pit_times, fastest_pit_team

In [42]:
def run_simulation(drivers, mu_r, sigma_r, mu_q, sigma_q, team_index, n_teams, p_dnf, grid_penalties,
                   track_position_weight, team_dnf_corr, team_pit_mu, team_pit_sigma, N=10000, seed=None):
    rng = np.random.default_rng(seed)
    n_drivers = len(drivers)

    grid, quali_result, q_draw = simulate_quali(mu_q, sigma_q, N, rng, grid_penalties)
    finish, dnf, r_draw = simulate_race(mu_r, sigma_r, grid, team_index, n_teams, p_dnf, track_position_weight, team_dnf_corr, N, rng, n_drivers)
    fl = simulate_fastest_lap(finish, r_draw, dnf, N, n_drivers)
    dotd = simulate_driver_of_the_day(finish, grid, dnf, N, n_drivers)
    pit_times, fastest_pit_team = simulate_pit_stops(team_index, n_teams, N, team_pit_mu, team_pit_sigma, rng)

    return {
        'grid': grid, 'quali_result': quali_result, 'finish': finish, 'dnf': dnf, 'fl': fl, 
        'dotd': dotd, 'pit_times': pit_times, 'fastest_pit_team': fastest_pit_team,
        'drivers': drivers, 'teams': teams,
    }

sim = run_simulation(
    drivers, mu_r, sigma_r, mu_q, sigma_q, team_index, n_teams,
    p_dnf=p_dnf_array,
    grid_penalties={'HAD': 5, 'COL': 15, 'LIN': 30},       # ex. {'ANT': 20} means 20 place grid penalty for antonelli
    track_position_weight=track_position_weight,
    team_dnf_corr=0.35,
    team_pit_mu=np.full(n_teams, 2.3), team_pit_sigma=np.full(n_teams, 0.4),
    N=10000
)

In [43]:
# grid position sanity: pole rate should roughly track qpace ranking
pole_rate = (sim['grid'] == 1).mean(axis=0)
print(pd.Series(pole_rate, index=drivers).sort_values(ascending=False).head())

# average grid position
avg_grid = np.nanmean(sim['grid'], axis=0)
print(pd.Series(avg_grid, index=drivers).sort_values())

# DNF rate should match your input p_dnf, roughly
realized_dnf_rate = sim['dnf'].mean(axis=0)
print(pd.DataFrame({'input': p_dnf_array, 'realized': realized_dnf_rate}, index=drivers))

# average finishing position should correlate with mu_r rank
avg_finish = np.nanmean(sim['finish'], axis=0)
print(pd.Series(avg_finish, index=drivers).sort_values())

# coverage check placeholder — once you have real historical outcomes to compare against
p5 = np.nanpercentile(sim['finish'], 5, axis=0)
p95 = np.nanpercentile(sim['finish'], 95, axis=0)
print(pd.DataFrame({'p5': p5, 'p95': p95}, index=drivers))

VER    0.3860
RUS    0.2360
LEC    0.1281
NOR    0.1176
ANT    0.1069
dtype: float64
VER     2.3809
RUS     2.9206
LEC     3.5981
NOR     3.7143
ANT     3.7539
HAM     5.2106
PIA     6.8337
LAW     8.9333
HAD     9.0174
GAS     9.5275
BOR    11.0165
HUL    11.5594
OCO    13.5100
ALO    14.1689
BEA    15.0418
SAI    15.9847
ALB    16.1636
STR    17.7198
BOT    19.0295
PER    19.9155
COL    21.0000
LIN    22.0000
dtype: float64
        input  realized
ALB  0.107912    0.1103
ALO  0.107912    0.1081
ANT  0.069882    0.0700
BEA  0.107912    0.1113
BOR  0.092251    0.0866
BOT  0.120172    0.1254
COL  0.053938    0.0564
GAS  0.069882    0.0725
HAD  0.092251    0.0929
HAM  0.069882    0.0684
HUL  0.114346    0.1160
LAW  0.082241    0.0799
LEC  0.092251    0.0926
LIN  0.077910    0.0809
NOR  0.100652    0.0973
OCO  0.069882    0.0690
PER  0.120172    0.1176
PIA  0.107912    0.1089
RUS  0.082241    0.0781
SAI  0.100652    0.1023
STR  0.130593    0.1271
VER  0.100652    0.1028
RUS     1.900423
V

In [14]:
import pickle

with open("simulation_results.pkl", "wb") as file:
    pickle.dump(sim, file)